### Parameter Initialization (Two-Layer Network)

This function sets up the initial parameters for a simple 2-layer neural network:

$$
X \; \rightarrow \; [W_1, b_1] \; \rightarrow \; \text{Hidden Layer (ReLU)} \; \rightarrow \; [W_2, b_2] \; \rightarrow \; \text{Output Layer (Sigmoid)}
$$

**Purpose:**  
To generate small random weights and zero biases — preventing symmetry and keeping activations within a stable numerical range.

**Implementation details:**
- $W_1 \in \mathbb{R}^{n_h \times n_x}$, $W_2 \in \mathbb{R}^{n_y \times n_h}$
  → Each column of \( W_1 \) connects all inputs to one hidden neuron.  
- Biases \( b_1, b_2 \) are initialized to zeros.  
- Multiplying by `0.01` ensures small weight magnitudes, preventing early saturation of activations.  
  *(Later layers use He/Xavier initialization instead.)*

**Sanity check:**
- Shapes:  
  `W1.shape = (n_h, n_x)`, `b1.shape = (n_h, 1)`  
  `W2.shape = (n_y, n_h)`, `b2.shape = (n_y, 1)`
- Reproducibility ensured via `np.random.seed(1)`.

**Quick recall (interview tip):**  
Symmetry breaking is crucial — if all weights start identical, all neurons will compute the same gradient.  
Biases can safely start at zero since they don’t affect symmetry.


In [ ]:
import numpy as np
def initialize_parameters(n_x, n_h, n_y):
    """
    Argument:
    n_x -- size of the input layer
    n_h -- size of the hidden layer
    n_y -- size of the output layer
    
    Returns:
    parameters -- python dictionary containing your parameters:
                    W1 -- weight matrix of shape (n_h, n_x)
                    b1 -- bias vector of shape (n_h, 1)
                    W2 -- weight matrix of shape (n_y, n_h)
                    b2 -- bias vector of shape (n_y, 1)
    """
    
    np.random.seed(1)
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters    

### Deep Network Parameter Initialization

This function initializes parameters for an $L$-layer neural network, where  
`layer_dims = [n^{[0]}, n^{[1]}, ..., n^{[L]}]` defines the number of units per layer.

Each layer $l$ has:
- Weight matrix $W^{[l]} \in \mathbb{R}^{n^{[l]} \times n^{[l-1]}}$
- Bias vector $b^{[l]} \in \mathbb{R}^{n^{[l]} \times 1}$

**Purpose:**  
To scale the two-layer initialization into a general framework for any number of layers.

**Implementation details:**
- Random small weights via `np.random.randn(...)*0.01` prevent symmetry across neurons.
- Biases are initialized to zeros.
- `np.random.seed(3)` ensures reproducibility.
- Assertions confirm that shapes match the intended layer dimensions.

**Key logic:**
```python
for l in range(1, L):
    parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.01
    parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))


Quick recall (interview tip):
Uniform initialization can lead to dead neurons or slow convergence.
We use small random values to start near zero but not identical — breaking symmetry while avoiding large activations.

In [2]:

def initialize_parameters_deep(layer_dims):
    """
    Arguments:
    layer_dims -- python array (list) containing the dimensions of each layer in our network
    
    Returns:
    parameters -- python dictionary containing your parameters "W1", "b1", ..., "WL", "bL":
                    Wl -- weight matrix of shape (layer_dims[l], layer_dims[l-1])
                    bl -- bias vector of shape (layer_dims[l], 1)
    """
    
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims) # number of layers in the network

    for l in range(1, L):

        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l - 1]) * 0.01
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
        
        assert(parameters['W' + str(l)].shape == (layer_dims[l], layer_dims[l - 1]))
        assert(parameters['b' + str(l)].shape == (layer_dims[l], 1))

        
    return parameters

### Linear Forward Step

Computes the affine transformation for one layer:

$$
Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}
$$

where:
- $A^{[l-1]}$ — activations from the previous layer (shape: $(n^{[l-1]}, m)$)  
- $W^{[l]}$ — weight matrix (shape: $(n^{[l]}, n^{[l-1]})$)  
- $b^{[l]}$ — bias vector (shape: $(n^{[l]}, 1)$)  
- $Z^{[l]}$ — linear output before the activation function

**Purpose:**  
Applies the linear transformation that projects the previous layer’s activations into the current layer’s space.

**Implementation details:**
- Uses `np.dot(W, A)` for matrix multiplication (efficient BLAS-level operation).  
- Bias `b` is broadcasted across all $m$ examples.  
- Returns both `Z` and a `cache` tuple `(A, W, b)` to reuse during backpropagation.

**Shape sanity check:**  
If `A_prev` has shape `(n_prev, m)` and `W` has shape `(n_curr, n_prev)`,  
then `Z` must have shape `(n_curr, m)`.

**Quick recall (interview tip):**  
This is the “linear” part of the common forward step pattern:
$$
A^{[l]} = g(Z^{[l]}) = g(W^{[l]} A^{[l-1]} + b^{[l]})
$$
Always verify matrix dimensions before activation to avoid silent broadcasting errors.


In [3]:
def linear_forward(A, W, b):
    """
    Implement the linear part of a layer's forward propagation.

    Arguments:
    A -- activations from previous layer (or input data): (size of previous layer, number of examples)
    W -- weights matrix: numpy array of shape (size of current layer, size of previous layer)
    b -- bias vector, numpy array of shape (size of the current layer, 1)

    Returns:
    Z -- the input of the activation function, also called pre-activation parameter 
    cache -- a python tuple containing "A", "W" and "b" ; stored for computing the backward pass efficiently
    """
    
    Z = np.dot(W, A) + b
    cache = (A, W, b)
    
    return Z, cache

### Linear → Activation Forward Step

Computes one full forward step for a layer:
$$
A^{[l]} = g(Z^{[l]}) = g(W^{[l]} A^{[l-1]} + b^{[l]})
$$
where $g(\cdot)$ is either the ReLU or sigmoid activation function.

**Purpose:**  
Wraps the linear transformation and activation into one reusable function for each layer.

**Implementation details:**
- Reuses `linear_forward()` to compute the pre-activation value $Z$.  
- Applies activation:
  - **Sigmoid:** $A = \sigma(Z) = \frac{1}{1 + e^{-Z}}$
  - **ReLU:** $A = \max(0, Z)$  
- Caches both:
  - `linear_cache = (A_prev, W, b)`  
  - `activation_cache = Z`  
  for use during backpropagation.

**Shape consistency:**
- Input `A_prev`: $(n^{[l-1]}, m)$  
- Output `A`: $(n^{[l]}, m)$

**Quick recall (interview tip):**  
Each layer performs:
1. Linear transform → $Z$
2. Activation → $A$
3. Store `(A_prev, W, b, Z)` for gradient computation later.

In deeper networks, this function is stacked for each layer, alternating ReLU for hidden layers and Sigmoid for the final layer (for binary classification).


In [4]:
def linear_activation_forward(A_prev, W, b, activation):
    """
    Implement the forward propagation for the LINEAR->ACTIVATION layer

    Arguments:
    A_prev -- activations from previous layer (or input data): (size of previous layer, number of examples)
    W -- weights matrix: numpy array of shape (size of current layer, size of previous layer)
    b -- bias vector, numpy array of shape (size of the current layer, 1)
    activation -- the activation to be used in this layer, stored as a text string: "sigmoid" or "relu"

    Returns:
    A -- the output of the activation function, also called the post-activation value 
    cache -- a python tuple containing "linear_cache" and "activation_cache";
             stored for computing the backward pass efficiently
    """
    
    if activation == "sigmoid":
        Z, linear_cache = linear_forward(A_prev, W, b)
        A = 1 / (1 + np.exp(-Z))
        activation_cache = Z

    
    elif activation == "relu":
        Z, linear_cache = linear_forward(A_prev, W, b)
        A = np.maximum(0, Z)
        activation_cache = Z

    cache = (linear_cache, activation_cache)

    return A, cache

### Full Forward Propagation: [LINEAR → ReLU] × (L−1) → [LINEAR → Sigmoid]

Performs forward propagation through the entire deep network:
$$
A^{[1]} = \text{ReLU}(W^{[1]}X + b^{[1]}), \quad
A^{[2]} = \text{ReLU}(W^{[2]}A^{[1]} + b^{[2]}), \dots, \quad
A^{[L]} = \sigma(W^{[L]}A^{[L-1]} + b^{[L]})
$$
The final activation $A^{[L]}$ (denoted `AL`) represents the model’s predicted probability for binary classification.

**Purpose:**  
Stacks multiple layers by repeatedly calling `linear_activation_forward()`:
- Uses **ReLU** activations for hidden layers.
- Uses **Sigmoid** activation for the output layer.

**Implementation details:**
- Loop over layers `1 → L-1` applying ReLU.  
- Final layer `L` applies Sigmoid.  
- Each layer’s `(linear_cache, activation_cache)` is stored in `caches` for backpropagation.

**Shape expectations:**
- Input `X`: $(n^{[0]}, m)$  
- Output `AL`: $(1, m)$ for binary classification

**Quick recall (interview tip):**
- Forward propagation builds the computation graph layer by layer.  
- Each cache contains the minimal info for backprop:  
  $(A^{[l-1]}, W^{[l]}, b^{[l]}, Z^{[l]})$.
- Always ensure `len(parameters) // 2 = L`, since each layer has both `W` and `b`.

This function forms the backbone of the training loop — once `AL` is obtained, it’s used to compute the cost and drive the backward pass.


In [5]:
def L_model_forward(X, parameters):
    """
    Forward propagation for the [LINEAR -> RELU]*(L-1) -> LINEAR -> SIGMOID network.

    Arguments:
        X : ndarray
            Input data of shape (n_x, m)
        parameters : dict
            Model parameters containing weights and biases for each layer

    Returns:
        AL : ndarray
            Output activation of the last layer
        caches : list
            List of caches from each layer for use in backpropagation
    """
    caches = []
    A = X
    L = len(parameters) // 2  # Number of layers in the network

    # Hidden layers: [LINEAR -> RELU] × (L-1)
    for l in range(1, L):
        A_prev = A
        A, cache = linear_activation_forward(
            A_prev,
            parameters[f"W{l}"],
            parameters[f"b{l}"],
            activation="relu"
        )
        caches.append(cache)

    # Output layer: [LINEAR -> SIGMOID]
    AL, cache = linear_activation_forward(
        A,
        parameters[f"W{L}"],
        parameters[f"b{L}"],
        activation="sigmoid"
    )
    caches.append(cache)

    return AL, caches


### Cost Function — Binary Cross-Entropy

Computes the loss between predictions `AL` and true labels `Y`:

$$
J = -\frac{1}{m}\sum_{i=1}^{m} \Big[ Y^{(i)} \log A_L^{(i)} + (1 - Y^{(i)}) \log(1 - A_L^{(i)}) \Big]
$$

**Purpose:**  
Measures how well the model’s predicted probabilities match the true labels.  
A lower cost indicates better alignment between prediction and ground truth.

**Implementation details:**
- Inputs:
  - `AL`: predicted probabilities $(1, m)$  
  - `Y`: true binary labels $(1, m)$  
- Uses elementwise log-loss and averages across all examples.  
- `np.squeeze` ensures the output is a scalar, not a 1×1 array.

**Quick recall (interview tip):**  
Cross-entropy penalizes confident wrong predictions heavily — ideal for classification tasks with sigmoid outputs.


In [6]:
def compute_cost(AL, Y):
    """
    Computes the binary cross-entropy cost.

    Arguments:
        AL : ndarray
            Probability vector from the model’s output layer, shape (1, m)
        Y : ndarray
            True binary labels, shape (1, m)

    Returns:
        cost : float
            Cross-entropy loss value
    """
    m = Y.shape[1]
    cost = -1 / m * np.sum(Y * np.log(AL) + (1 - Y) * np.log(1 - AL))
    cost = np.squeeze(cost)  # Ensure scalar output
    return cost


### Linear Backward Step

Computes gradients for the linear component of layer $l$:

$$
Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}
$$

During backpropagation, given the gradient $dZ^{[l]}$, we compute:

$$
\begin{aligned}
dW^{[l]} &= \frac{1}{m} dZ^{[l]} (A^{[l-1]})^T, \\
db^{[l]} &= \frac{1}{m} \sum_i dZ^{[l]}_i, \\
dA^{[l-1]} &= (W^{[l]})^T dZ^{[l]}.
\end{aligned}
$$

**Purpose:**  
Propagate the gradient of the cost function backward through the linear transformation of a given layer.

**Implementation details:**
- `dZ`: upstream gradient from the activation derivative  
- `cache`: tuple `(A_prev, W, b)` saved from the forward pass  
- Computes partial derivatives with respect to parameters and the previous layer’s activation.  
- Divides by `m` to average over the batch.  
- Keeps dimensions consistent via `keepdims=True` in `np.sum`.

**Quick recall (interview tip):**  
This is the gradient chain rule for the affine map.  
Always check that:
- `dW.shape == W.shape`  
- `db.shape == b.shape`  
- `dA_prev.shape == A_prev.shape`


In [8]:
def linear_backward(dZ, cache):
    """
    Computes gradients for the linear portion of one layer.

    Arguments:
        dZ : ndarray
            Gradient of the cost with respect to the linear output Z of the current layer
        cache : tuple
            Cached values (A_prev, W, b) from the forward pass

    Returns:
        dA_prev : ndarray
            Gradient with respect to the previous layer’s activation
        dW : ndarray
            Gradient with respect to the current layer’s weights
        db : ndarray
            Gradient with respect to the current layer’s biases
    """
    A_prev, W, b = cache
    m = A_prev.shape[1]

    dW = (1 / m) * np.dot(dZ, A_prev.T)
    db = (1 / m) * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = np.dot(W.T, dZ)

    return dA_prev, dW, db


### Linear → Activation Backward Step

Computes the gradient for one layer that includes both linear and activation components.

For layer $l$:
$$
A^{[l]} = g(Z^{[l]}) = g(W^{[l]}A^{[l-1]} + b^{[l]})
$$

Given the gradient $dA^{[l]}$ (upstream), we compute:
1. Derivative of activation:
   - **ReLU:** 
     $$
     dZ^{[l]} = dA^{[l]} \odot \mathbb{1}_{Z^{[l]} > 0}
     $$
   - **Sigmoid:** 
     $$
     s = \sigma(Z^{[l]}) = \frac{1}{1 + e^{-Z^{[l]}}}, \quad
     dZ^{[l]} = dA^{[l]} \cdot s \cdot (1 - s)
     $$
2. Then apply the linear backward step:

   $$
   (dA^{[l-1]}, dW^{[l]}, db^{[l]}) = {linear_backward}(dZ^{[l]}, (A^{[l-1]}, W^{[l]}, b^{[l]}))
   $$

**Purpose:**  
Combine the derivative of the activation with the linear gradient to propagate the error backward through a single layer.

**Implementation notes:**
- `activation_cache` stores $Z^{[l]}$ for reuse in derivative computations.
- ReLU zeroes gradients for non-active neurons.
- Sigmoid derivative $s(1-s)$ ensures smooth gradient flow for probabilistic outputs.

**Quick recall (interview tip):**  
Each layer’s backward pass = activation derivative × linear derivative.  
ReLU is sparse (good for deep networks), while Sigmoid saturates (use it only for outputs).


In [9]:
def linear_activation_backward(dA, cache, activation):
    """
    Backward propagation for a layer with a specified activation.

    Arguments:
        dA : ndarray
            Post-activation gradient for the current layer
        cache : tuple
            (linear_cache, activation_cache) from the forward pass
        activation : str
            Activation function name ("relu" or "sigmoid")

    Returns:
        dA_prev : ndarray
            Gradient with respect to the previous layer’s activation
        dW : ndarray
            Gradient with respect to the current layer’s weights
        db : ndarray
            Gradient with respect to the current layer’s biases
    """
    linear_cache, activation_cache = cache

    if activation == "relu":
        dZ = np.array(dA, copy=True)
        dZ[activation_cache <= 0] = 0
        dA_prev, dW, db = linear_backward(dZ, linear_cache)

    elif activation == "sigmoid":
        Z = activation_cache
        s = 1 / (1 + np.exp(-Z))
        dZ = dA * s * (1 - s)
        dA_prev, dW, db = linear_backward(dZ, linear_cache)

    return dA_prev, dW, db


### Full Backward Propagation — [LINEAR ← ReLU] × (L − 1) ← [LINEAR ← Sigmoid]

Implements the complete backward pass through the entire deep network.  
Starting from the output layer, the algorithm iteratively computes gradients for each layer in reverse order.

For the final layer $L$:
$$
dZ^{[L]} = A^{[L]} - Y
$$
(since the derivative of cross-entropy loss combined with a sigmoid activation simplifies neatly).

For layers $l = L-1, \dots, 1$:
$$
(dA^{[l]}, dW^{[l+1]}, db^{[l+1]}) = 
\text{linear\_activation\_backward}(dA^{[l+1]}, \text{cache}^{[l+1]}, \text{"relu"})
$$

**Purpose:**  
Propagate gradients from the output back through all layers, updating:
- `grads["dA" + str(l)]` — upstream gradient for each layer  
- `grads["dW" + str(l)]` — weight gradients  
- `grads["db" + str(l)]` — bias gradients  

**Implementation details:**
- The final layer uses `"sigmoid"` activation (output probability).  
- All hidden layers use `"relu"`.  
- Gradients are stored in a dictionary for parameter updates later.  

**Shape sanity checks:**  
If `AL.shape = (1, m)`, then  
`dW[l].shape = W[l].shape` and `db[l].shape = b[l].shape` for all $l$.

**Quick recall (interview tip):**  
- The backprop chain alternates between applying the activation derivative and the linear derivative.  
- Always iterate backward (`reversed(range(...))`) and cache all gradients for later optimization.


In [10]:
def L_model_backward(AL, Y, caches):
    """
    Backward propagation for the entire [LINEAR -> RELU]*(L-1) -> LINEAR -> SIGMOID network.

    Arguments:
        AL : ndarray
            Output from the forward pass (final activation), shape (1, m)
        Y : ndarray
            True labels vector, shape (1, m)
        caches : list
            List of caches from the forward pass for each layer

    Returns:
        grads : dict
            Dictionary containing gradients for every parameter and activation
    """
    grads = {}
    L = len(caches)            # Number of layers
    m = AL.shape[1]
    Y = Y.reshape(AL.shape)    # Ensure same shape as AL

    # Initialize backpropagation from output layer
    dAL = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL))

    # Layer L (SIGMOID)
    current_cache = caches[-1]
    grads["dA" + str(L - 1)], grads["dW" + str(L)], grads["db" + str(L)] = linear_activation_backward(
        dAL, current_cache, activation="sigmoid"
    )

    # Loop from L-1 down to 1 (RELU)
    for l in reversed(range(L - 1)):
        current_cache = caches[l]
        dA_prev_temp, dW_temp, db_temp = linear_activation_backward(
            grads["dA" + str(l + 1)], current_cache, activation="relu"
        )
        grads["dA" + str(l)] = dA_prev_temp
        grads["dW" + str(l + 1)] = dW_temp
        grads["db" + str(l + 1)] = db_temp

    return grads


### Parameter Update — Gradient Descent Step

Updates all parameters using the gradients computed during backpropagation.

For each layer $l$:
$$
\begin{aligned}
W^{[l]} &:= W^{[l]} - \alpha \, dW^{[l]} \\
b^{[l]} &:= b^{[l]} - \alpha \, db^{[l]}
\end{aligned}
$$

where:
- $\alpha$ is the **learning rate**, controlling the step size.  
- $dW^{[l]}$, $db^{[l]}$ come from backpropagation.  

**Purpose:**  
Implements the simplest optimization rule — batch gradient descent.  
It reduces the cost function by moving parameters opposite to the gradient direction.

**Implementation details:**
- Iterates through all layers `1 → L`.  
- Uses the gradient dictionary from `L_model_backward`.  
- Returns updated parameter dictionary for the next iteration.

**Quick recall (interview tip):**  
- Gradient descent guarantees cost reduction if $\alpha$ is small enough.  
- Too small → slow convergence; too large → oscillation or divergence.


In [11]:
def update_parameters(parameters, grads, learning_rate):
    """
    Updates parameters using gradient descent.

    Arguments:
        parameters : dict
            Current parameters of the model (W[l], b[l])
        grads : dict
            Gradients computed from backpropagation (dW[l], db[l])
        learning_rate : float
            Step size for the gradient update

    Returns:
        parameters : dict
            Updated parameters after one optimization step
    """
    L = len(parameters) // 2  # number of layers

    for l in range(1, L + 1):
        parameters[f"W{l}"] -= learning_rate * grads[f"dW{l}"]
        parameters[f"b{l}"] -= learning_rate * grads[f"db{l}"]

    return parameters
